# RNNs & Sequence Modeling: LSTM, GRU, Attention

**Module 4 - Notebook 3** | Skillsoft Track 2: DL Programmer

## Objectives
- Implement LSTM/GRU from scratch
- Text classification with bidirectional RNNs
- Attention mechanism visualization
- Seq2Seq with teacher forcing

In [ ]:
# Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchtext
from torchtext.datasets import IMDB
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. LSTM/GRU from Scratch

In [ ]:
class ScratchLSTMCell(nn.Module):
    """LSTM cell implemented manually."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.W_ii = nn.Linear(input_size, hidden_size)
        self.W_if = nn.Linear(input_size, hidden_size)
        self.W_ig = nn.Linear(input_size, hidden_size)
        self.W_io = nn.Linear(input_size, hidden_size)
        self.W_hi = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W_hf = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W_hg = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W_ho = nn.Linear(hidden_size, hidden_size, bias=False)
    
    def forward(self, x, hx=None):
        if hx is None:
            h = torch.zeros(x.size(0), self.hidden_size, device=x.device)
            c = torch.zeros(x.size(0), self.hidden_size, device=x.device)
        else:
            h, c = hx
        
        i = torch.sigmoid(self.W_ii(x) + self.W_hi(h))
        f = torch.sigmoid(self.W_if(x) + self.W_hf(h))
        g = torch.tanh(self.W_ig(x) + self.W_hg(h))
        o = torch.sigmoid(self.W_io(x) + self.W_ho(h))
        
        c = f * c + i * g
        h = o * torch.tanh(c)
        return h, c

class ScratchGRUCell(nn.Module):
    """GRU cell implemented manually."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.W_ir = nn.Linear(input_size, hidden_size)
        self.W_iz = nn.Linear(input_size, hidden_size)
        self.W_in = nn.Linear(input_size, hidden_size)
        self.W_hr = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W_hz = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W_hn = nn.Linear(hidden_size, hidden_size, bias=False)
    
    def forward(self, x, h=None):
        if h is None:
            h = torch.zeros(x.size(0), self.hidden_size, device=x.device)
        
        r = torch.sigmoid(self.W_ir(x) + self.W_hr(h))
        z = torch.sigmoid(self.W_iz(x) + self.W_hz(h))
        n = torch.tanh(self.W_in(x) + r * self.W_hn(h))
        h = (1 - z) * n + z * h
        return h

# Test cells
lstm = ScratchLSTMCell(10, 20).to(device)
gru = ScratchGRUCell(10, 20).to(device)
x = torch.randn(4, 10).to(device)
h, c = lstm(x)
h_gru = gru(x)
print(f"LSTM output: {h.shape}, cell: {c.shape}")
print(f"GRU output: {h_gru.shape}")

## 2. Text Classification with Bidirectional LSTM

In [ ]:
# IMDB sentiment classification
tokenizer = get_tokenizer('basic_english')

def yield_tokens(data_iter):
    for label, text in data_iter:
        yield tokenizer(text)

train_iter = IMDB(split='train')
vocab = build_vocab_from_iterator(yield_tokens(train_iter), specials=["<unk>", "<pad>"])
vocab.set_default_index(vocab["<unk>"])

text_pipeline = lambda x: vocab(tokenizer(x))
label_pipeline = lambda x: 1 if x == 'pos' else 0

def collate_batch(batch):
    labels, texts = zip(*batch)
    label_list = torch.tensor([label_pipeline(l) for l in labels], dtype=torch.long)
    text_list = [torch.tensor(text_pipeline(t), dtype=torch.long) for t in texts]
    lengths = torch.tensor([len(t) for t in text_list])
    text_padded = nn.utils.rnn.pad_sequence(text_list, batch_first=True, padding_value=vocab["<pad>"])
    return text_padded, label_list, lengths

train_dataset = list(IMDB(split='train'))
test_dataset = list(IMDB(split='test'))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_batch)

print(f"Vocab size: {len(vocab)}")
print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, num_classes, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=vocab["<pad>"])
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers,
            bidirectional=True, batch_first=True, dropout=dropout
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
    
    def forward(self, x, lengths):
        embedded = self.dropout(self.embedding(x))
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, (hidden, cell) = self.lstm(packed)
        # Concatenate final forward and backward hidden states
        hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.fc(self.dropout(hidden))

model = BiLSTMClassifier(
    vocab_size=len(vocab), embed_dim=100, hidden_dim=128,
    num_layers=2, num_classes=2, dropout=0.5
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for text, labels, lengths in loader:
        text, labels, lengths = text.to(device), labels.to(device), lengths.to(device)
        optimizer.zero_grad()
        output = model(text, lengths)
        loss = criterion(output, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total

# Train a few epochs
for epoch in range(3):
    loss, acc = train_epoch(model, train_loader, criterion, optimizer, device)
    print(f"Epoch {epoch}: Loss={loss:.4f}, Acc={acc:.4f}")

## 3. Attention Mechanism Visualization

In [ ]:
class Attention(nn.Module):
    """Additive attention (Bahdanau)."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.W = nn.Linear(hidden_dim * 2, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)
    
    def forward(self, hidden, encoder_outputs):
        # hidden: (batch, hidden*2) - decoder hidden
        # encoder_outputs: (batch, seq_len, hidden*2)
        batch, seq_len, _ = encoder_outputs.size()
        hidden = hidden.unsqueeze(1).repeat(1, seq_len, 1)
        energy = torch.tanh(self.W(torch.cat([hidden, encoder_outputs], dim=2)))
        attention = self.v(energy).squeeze(2)
        return F.softmax(attention, dim=1)

class Seq2SeqAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers):
        super().__init__()
        self.encoder_embedding = nn.Embedding(vocab_size, embed_dim)
        self.encoder = nn.LSTM(embed_dim, hidden_dim, num_layers, bidirectional=True, batch_first=True)
        self.attention = Attention(hidden_dim)
        self.decoder_embedding = nn.Embedding(vocab_size, embed_dim)
        self.decoder = nn.LSTM(embed_dim + hidden_dim * 2, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch, trg_len = trg.shape
        outputs = torch.zeros(batch, trg_len, vocab_size).to(device)
        
        # Encode
        embedded = self.encoder_embedding(src)
        encoder_outputs, (hidden, cell) = self.encoder(embedded)
        
        # Decoder init
        hidden = torch.cat([hidden[-2:].transpose(0,1).reshape(batch, -1),
                          cell[-2:].transpose(0,1).reshape(batch, -1)], dim=1)
        input = trg[:, 0].unsqueeze(1)  # <sos>
        
        for t in range(1, trg_len):
            embedded = self.decoder_embedding(input)
            attn_weights = self.attention(hidden, encoder_outputs)
            context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)
            decoder_input = torch.cat([embedded.squeeze(1), context], dim=1).unsqueeze(1)
            output, (hidden, cell) = self.decoder(decoder_input, (hidden.unsqueeze(0), cell.unsqueeze(0)))
            pred = self.fc(output.squeeze(1))
            outputs[:, t] = pred
            
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            input = trg[:, t].unsqueeze(1) if teacher_force else pred.argmax(1).unsqueeze(1)
        
        return outputs

print("Seq2Seq with Attention defined")

## 4. Teacher Forcing vs Scheduled Sampling